# Quantitative Portfolio Research: Data Exploration
This notebook analyzes historical adjusted closing prices, daily return distributions, correlations, and sector breakdowns across our 12 large-cap US equities.

In [ ]:
import sys
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Robust project root discovery
current_path = pathlib.Path('.').resolve()
project_root = current_path if (current_path / 'src').exists() else current_path.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TICKERS, ASSET_INFO, CLEAN_PRICES_PATH, EXPORTS_DIR

sns.set_theme(style='whitegrid', palette='husl')
print(f"Project root: {project_root}")

## 1. Load Clean Market Prices and Asset Returns

In [ ]:
prices_path = project_root / 'data' / 'processed' / 'prices_clean.csv'
returns_path = project_root / 'data' / 'exports' / 'asset_returns.csv'

prices_df = pd.read_csv(prices_path, index_col=0, parse_dates=True)
returns_df = pd.read_csv(returns_path)

print("--- Dataset Overview ---")
print(f"Prices shape: {prices_df.shape} (Dates x Tickers)")
print(f"Date range: {prices_df.index.min().date()} to {prices_df.index.max().date()}")
print(f"Universe: {list(prices_df.columns)}")

## 2. Data Quality & Missing Value Audit

In [ ]:
missing = prices_df.isna().sum()
print("Missing values per column:")
print(missing)

plt.figure(figsize=(10, 3))
sns.heatmap(prices_df.isna().T, cbar=False, cmap='viridis')
plt.title("Missing Value Map across 1,509 Trading Days (0 Missing)")
plt.xlabel("Trading Day Index")
plt.tight_layout()
plt.show()

## 3. Normalized Price Trajectories (Base 100)

In [ ]:
normalized_prices = (prices_df / prices_df.iloc[0]) * 100

plt.figure(figsize=(14, 7))
for col in normalized_prices.columns:
    plt.plot(normalized_prices.index, normalized_prices[col], label=col, alpha=0.85)

plt.title("Normalized Asset Growth (2019-2024, Base = 100)", fontsize=14, fontweight='bold')
plt.xlabel("Date", fontsize=11)
plt.ylabel("Normalized Value ($)", fontsize=11)
plt.legend(loc='upper left', bbox_to_anchor=(1.01, 1), title="Ticker")
plt.tight_layout()
plt.show()

## 4. Daily Return Distributions

In [ ]:
daily_returns = prices_df.pct_change().dropna()

fig, axes = plt.subplots(4, 3, figsize=(15, 12), sharex=True)
axes = axes.flatten()

for i, ticker in enumerate(prices_df.columns):
    sns.histplot(daily_returns[ticker], kde=True, ax=axes[i], color='teal', bins=40, stat='density')
    axes[i].set_title(f"{ticker} ({ASSET_INFO.get(ticker, {}).get('name', ticker)})", fontsize=10)
    axes[i].set_xlabel("Daily Return")

plt.suptitle("Daily Return Distributions (Empirical vs Normal Kernel)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Statistical Summary Table

In [ ]:
stats_df = pd.DataFrame({
    'Mean Daily (%)': daily_returns.mean() * 100,
    'Daily Volatility (%)': daily_returns.std() * 100,
    'Annualized Volatility (%)': daily_returns.std() * np.sqrt(252) * 100,
    'Skewness': daily_returns.skew(),
    'Kurtosis': daily_returns.kurtosis(),
    'Min Return (%)': daily_returns.min() * 100,
    'Max Return (%)': daily_returns.max() * 100,
})
display(stats_df.round(3)) if 'display' in globals() else print(stats_df.round(3))

## 6. Return Correlation Matrix Heatmap

In [ ]:
corr_matrix = daily_returns.corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-0.2, vmax=1.0, linewidths=0.5)
plt.title("Asset Daily Return Correlation Matrix (2019-2024)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Individual Asset Cumulative Total Returns

In [ ]:
cum_returns = (1 + daily_returns).cumprod() - 1

plt.figure(figsize=(14, 6))
final_rets = (cum_returns.iloc[-1] * 100).sort_values(ascending=False)
bars = plt.bar(final_rets.index, final_rets.values, color=sns.color_palette('viridis', len(final_rets)))
plt.title("Total Cumulative Return per Stock (2019-2024)", fontsize=14, fontweight='bold')
plt.ylabel("Total Return (%)")
plt.xticks(rotation=45)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{yval:.0f}%", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Sector Representation Breakdown

In [ ]:
sector_counts = pd.Series([info['sector'] for info in ASSET_INFO.values()]).value_counts()

plt.figure(figsize=(8, 4))
sns.barplot(x=sector_counts.values, y=sector_counts.index, palette='Blues_r')
plt.title("Asset Universe Breakdown by Sector (12 Stocks across 7 Sectors)", fontsize=12, fontweight='bold')
plt.xlabel("Number of Assets")
plt.tight_layout()
plt.show()